---
## 4. Criação das Métricas de Tempo (KPIs Logísticos)

Essas métricas são o coração da análise de SLA. São calculadas a partir das colunas de data dos pedidos entregues.

| Métrica | Fórmula | Unidade |
|---|---|---|
| `tempo_aprovacao_h` | `approved_at` − `purchase_timestamp` | horas |
| `tempo_entrega_dias` | `delivered_customer` − `purchase_timestamp` | dias |
| `tempo_estimado_dias` | `estimated_delivery` − `purchase_timestamp` | dias |
| `atraso_dias` | `delivered_customer` − `estimated_delivery` | dias |
| `atrasado` | `atraso_dias > 0` | booleano |

In [4]:
import pandas as pd


In [5]:
df_olist_customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
df_olist_geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
olist_order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
olist_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
olist_products = pd.read_csv("../data/raw/olist_products_dataset.csv")
olist_sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_category_name_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")
df_orders_delivered = pd.read_pickle("../data/processed/df_orders_delivered.pkl")
df_reviews_clean = pd.read_pickle("../data/processed/df_reviews_clean.pkl")

In [6]:
# Métricas de tempo
df_orders_delivered['tempo_aprovacao_h'] = (
    df_orders_delivered['order_approved_at'] -
    df_orders_delivered['order_purchase_timestamp']
).dt.total_seconds() / 3600

df_orders_delivered['tempo_entrega_dias'] = (
    df_orders_delivered['order_delivered_customer_date'] -
    df_orders_delivered['order_purchase_timestamp']
).dt.days

df_orders_delivered['tempo_estimado_dias'] = (
    df_orders_delivered['order_estimated_delivery_date'] -
    df_orders_delivered['order_purchase_timestamp']
).dt.days

df_orders_delivered['atraso_dias'] = (
    df_orders_delivered['order_delivered_customer_date'] -
    df_orders_delivered['order_estimated_delivery_date']
).dt.days

df_orders_delivered['atrasado'] = (df_orders_delivered['atraso_dias'] > 0).astype(int)

---
## Validação e Consistência dos Dados

Etapa crítica para garantir a confiabilidade das análises. Verificamos:
1. **Inconsistências temporais**: entregas antes da compra (fisicamente impossível)
2. **Outliers de tempo**: entregas com mais de 180 dias (provavelmente erros)
3. **Cobertura do join**: quantos pedidos terão avaliação disponível
4. **Pedidos sem vendedor ou produto**: impacto nas análises por seller/categoria

In [7]:
# ============================================================
# VALIDAÇÃO DE CONSISTÊNCIA TEMPORAL
# ============================================================

# 1. Entregas antes da compra — impossível, erro nos dados
entrega_antes_compra = df_orders_delivered[
    df_orders_delivered['order_delivered_customer_date'] <
    df_orders_delivered['order_purchase_timestamp']
]
print(f'⚠️  Entregas com data anterior à compra:    {len(entrega_antes_compra):,}')

# 2. Tempos negativos (erro de data) ou absurdos (> 180 dias)
tempos_invalidos = df_orders_delivered[
    (df_orders_delivered['tempo_entrega_dias'] < 0) |
    (df_orders_delivered['tempo_entrega_dias'] > 180)
]
print(f'⚠️  Tempos de entrega inválidos (<0 ou >180 dias): {len(tempos_invalidos):,}')

# ============================================================
# REMOÇÃO DAS INCONSISTÊNCIAS
# ============================================================
# Removemos os registros problemáticos para não distorcer
# as métricas de tempo médio, histogramas e correlações.
# ============================================================

df_orders_clean = df_orders_delivered[
    (df_orders_delivered['tempo_entrega_dias'] >= 0) &
    (df_orders_delivered['tempo_entrega_dias'] <= 180) &
    (df_orders_delivered['order_delivered_customer_date'] >=
     df_orders_delivered['order_purchase_timestamp'])
].copy()

removidos = len(df_orders_delivered) - len(df_orders_clean)
print(f'\n✅ Base orders limpa: {len(df_orders_clean):,} pedidos')
print(f'   Registros removidos por inconsistência: {removidos:,}')

# ============================================================
# VALIDAÇÃO DE COBERTURA DOS JOINS
# ============================================================

# Pedidos que terão avaliação após o join
com_review = df_orders_clean['order_id'].isin(df_reviews_clean['order_id']).sum()
pct_review  = com_review / len(df_orders_clean) * 100
print(f'\n📊 Cobertura de avaliações (reviews):')
print(f'   Pedidos com avaliação disponível: {com_review:,} ({pct_review:.1f}%)')
print(f'   Pedidos sem avaliação:            {len(df_orders_clean)-com_review:,} ({100-pct_review:.1f}%)')

⚠️  Entregas com data anterior à compra:    0
⚠️  Tempos de entrega inválidos (<0 ou >180 dias): 14

✅ Base orders limpa: 96,456 pedidos
   Registros removidos por inconsistência: 22

📊 Cobertura de avaliações (reviews):
   Pedidos com avaliação disponível: 95,811 (99.3%)
   Pedidos sem avaliação:            645 (0.7%)


In [8]:
df_orders_clean.to_pickle("../data/processed/df_orders_clean.pkl")